In [ ]:
# Construção do modelo de aprendizado supervisionado com todos os aprendizados da aula 1 à 5 (preparatório para prova)
# Objetivo: consolidar todo o pipeline de Natural Language Understanding (NLU) construído até aqui.

#BLOCO 1: Preparação do Ambiente e Carga do Dataset

#Instalos as bibliotecas necessárias para este algoritmo. Fazer em janela separada ou em linha de comando
#!pip install gensim
#!python -m spacy download pt_core_news_sm

#Importamos as ferramentas de processamento e definimos nosso dataset focado no atendimento imobiliário (vendas, aluguel, suporte e contratos).
import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
from sklearn.tree import DecisionTreeClassifier
import gradio as gr


# ============================================================
# 1. CARREGAMENTO DO MODELO DE LINGUAGEM EM PORTUGUÊS
# ============================================================

print("Carregando modelo morfológico SpaCy (pt_core_news_sm)")

nlp = spacy.load("pt_core_news_sm")


# 2. CARREGAMENTO DOS WORD EMBEDDINGS

print("Carregando espaço vetorial denso de embeddings modelo GloVe")

word_vectors = api.load("glove-wiki-gigaword-50")


# 3. BASE DE DADOS SUPERVISIONADA

dados_imobiliaria = [

    # Intenção: comprar_imovel

    ("Quero comprar um apartamento de 3 quartos com varanda",
     "comprar_imovel"),

    ("Gostaria de ver casas à venda no centro da cidade",
     "comprar_imovel"),

    ("Qual o preço médio para compra de cobertura com piscina?",
     "comprar_imovel"),

    ("Procuro imóvel residencial para comprar com financiamento",
     "comprar_imovel"),

    ("Vocês têm sobrados à venda na zona sul?",
     "comprar_imovel"),


    # Intenção: alugar_imovel

    ("Procurando kitnet para alugar perto da faculdade",
     "alugar_imovel"),

    ("Qual o valor do aluguel deste apartamento de 2 dormitórios?",
     "alugar_imovel"),

    ("Quero alugar um galpão comercial para minha empresa",
     "alugar_imovel"),

    ("Quais imóveis estão disponíveis para locação imediata?",
     "alugar_imovel"),

    ("Preciso de uma casa para alugar que aceite animais",
     "alugar_imovel"),


    # Intenção: suporte_manutencao

    ("O chuveiro do apartamento alugado queimou, como pedir conserto?",
     "suporte_manutencao"),

    ("Muro da casa está com infiltração e vazamento de água",
     "suporte_manutencao"),

    ("Preciso do contato do encanador para reparo na cozinha",
     "suporte_manutencao"),

    ("A porta da varanda quebrou, quem faz a manutenção?",
     "suporte_manutencao"),

    ("Vazamento no teto do banheiro precisa de reparo urgente",
     "suporte_manutencao"),


    # Intenção: 2via_boleto_contrato

    ("Como faço para baixar a segunda via do boleto do aluguel?",
     "2via_boleto_contrato"),

    ("Não recebi o boleto deste mês para pagamento",
     "2via_boleto_contrato"),

    ("Preciso do informe de rendimentos e cópia do contrato",
     "2via_boleto_contrato"),

    ("Onde pego o boleto atualizado com o valor do condomínio?",
     "2via_boleto_contrato"),

    ("Quero solicitar a segunda via do recibo de pagamento",
     "2via_boleto_contrato")
]


# Criar DataFrame
df = pd.DataFrame(
    dados_imobiliaria,
    columns=["mensagem", "intencao"]
)


print(
    f"Dataset carregado com {len(df)} mensagens "
    f"divididas em {df['intencao'].nunique()} intenções."
)



# BLOCO 2: ESTEIRA NLU
# PRÉ-PROCESSAMENTO E VETORIZAÇÃO


def preprocessar_texto(texto: str) -> str:
    """
    Realiza a limpeza e padronização do texto:

    1. Converte para letras minúsculas
    2. Remove números e caracteres especiais
    3. Remove stop words
    4. Extrai os lemas das palavras usando SpaCy
    """

    # 1. Converter para minúsculas

    texto_limpo = texto.lower()


    # 2. Remover números e caracteres especiais

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )



    # 3. Processamento com SpaCy

    doc = nlp(texto_limpo)


    # 4. Lematização + remoção de stop words

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1
    ]


    # 5. Reconstruir a frase

    return " ".join(tokens)



def extrair_sentence_embedding(
    texto_limpo: str,
    modelo_emb
) -> np.ndarray:

    """
    Converte uma frase limpa em um único vetor
    usando Mean Pooling dos embeddings das palavras.
    """

    # Separar as palavras
    palavras = texto_limpo.split()


    # Buscar os vetores das palavras existentes no modelo
    vetores = [
        modelo_emb[p]
        for p in palavras
        if p in modelo_emb
    ]


    # Caso nenhuma palavra seja encontrada

    if len(vetores) == 0:

        return np.zeros(
            modelo_emb.vector_size
        )


    # Mean Pooling

    return np.mean(
        vetores,
        axis=0
    )


# APLICAÇÃO DO PRÉ-PROCESSAMENTO NO DATASET

df["mensagem_limpa"] = df["mensagem"].apply(
    preprocessar_texto
)


# CONSTRUÇÃO DA MATRIZ X

X_densos = np.array([
    extrair_sentence_embedding(
        txt,
        word_vectors
    )
    for txt in df["mensagem_limpa"]
])


# CONSTRUÇÃO DO VETOR Y

y = df["intencao"].values


# Conferir dimensões
print(
    "Dimensão da matriz X:",
    X_densos.shape
)

print(
    "Dimensão do vetor y:",
    y.shape
)


# BLOCO 3: TREINAMENTO DO MODELO
# ÁRVORE DE DECISÃO


# LAB 01
# Substituição da Regressão Logística por Decision Tree

modelo_nlu = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)


# Treinamento supervisionado

modelo_nlu.fit(
    X_densos,
    y
)


print("Modelo supervisionado treinado com Árvore de Decisão!")


# BASE DE CONHECIMENTO
# RESPOSTAS PADRÃO DO SAC

RESPOSTAS_PADRAO = {

    "comprar_imovel": (
        "**Atendimento de Vendas:** "
        "Ficamos felizes com seu interesse! "
        "Você pode conferir nosso catálogo de imóveis à venda "
        "em nosso site www.imobiliaria.com/vendas "
        "ou aguardar que um de nossos corretores "
        "entrará em contato em instantes."
    ),


    "alugar_imovel": (
        "**Atendimento de Locação:** "
        "Temos ótimas opções disponíveis! "
        "Acesse www.imobiliaria.com/aluguel "
        "para filtrar por região e valor. "
        "Para agendar uma visita, envie o código "
        "do imóvel por aqui."
    ),


    "suporte_manutencao": (
        "**Suporte e Manutenção:** "
        "Sentimos muito pelo inconveniente. "
        "Por favor, abra um chamado urgente em nosso "
        "portal do inquilino "
        "(www.imobiliaria.com/manutencao) "
        "anexando fotos ou vídeos do problema "
        "para acionarmos nossos prestadores."
    ),


    "2via_boleto_contrato": (
        "**Financeiro e Contratos:** "
        "Para acessar boletos ou documentos, "
        "acesse a Área do Cliente em "
        "www.imobiliaria.com/cliente "
        "informando seu CPF e senha. "
        "Lá você baixa a 2ª via atualizada em segundos."
    )
}


# BLOCO 4: MOTOR DE INFERÊNCIA
# COM LÓGICA DE FALLBACK


# Limiar mínimo de confiança
LIMIAR_CONFIANCA = 0.50



def processar_atendimento_sac(
    mensagem_usuario: str
):


    # VERIFICAR SE O USUÁRIO DIGITOU ALGUMA MENSAGEM

    if (
        not mensagem_usuario
        or not mensagem_usuario.strip()
    ):

        return (
            "N/A",
            "0.0%",
            "Aguardando mensagem...",
            "Aguardando entrada do usuário..."
        )


    # STEP 1 — PRÉ-PROCESSAMENTO

    msg_limpa = preprocessar_texto(
        mensagem_usuario
    )



    # STEP 2 — VETORIZAÇÃO

    vetor_input = extrair_sentence_embedding(
        msg_limpa,
        word_vectors
    ).reshape(1, -1)


    # STEP 3 — PREDIÇÃO DAS PROBABILIDADES


    probabilidades = (
        modelo_nlu
        .predict_proba(vetor_input)[0]
    )


    # Encontrar índice da maior probabilidade
    idx_maior_prob = np.argmax(
        probabilidades
    )


    # Recuperar confiança
    confianca = probabilidades[
        idx_maior_prob
    ]


    # Recuperar intenção
    intencao_detectada = (
        modelo_nlu.classes_[
            idx_maior_prob
        ]
    )


    # Converter confiança para porcentagem
    percentual_confianca = (
        f"{confianca * 100:.1f}%"
    )


    # STEP 4 — REGRA DE DECISÃO / FALLBACK

    if confianca >= LIMIAR_CONFIANCA:

        classificacao_status = (
            f"IDENTIFICADO ({intencao_detectada})"
        )

        texto_resposta = (
            RESPOSTAS_PADRAO[
                intencao_detectada
            ]
        )

    else:

        classificacao_status = (
            "UNCERTAIN (Fallback Acionado)"
        )

        texto_resposta = (
            "Desculpe, não consegui compreender "
            "com clareza a sua solicitação. "
            "Estou transferindo agora mesmo sua "
            "conversa para um de nossos atendentes. "
            "Por favor, aguarde um momento."
        )


    # CARD HTML DA RESPOSTA

    card_resposta = f"""

    <div style="
        background-color: #f0f4f9;
        border-left: 5px solid #2b5c8f;
        padding: 15px;
        border-radius: 8px;
        margin-top: 10px;
    ">

        <h4 style="
            margin: 0 0 8px 0;
            color: #2b5c8f;
        ">
            Resposta Automática do SAC:
        </h4>

        <p style="
            margin: 0;
            font-size: 15px;
            color: #1a1a1a;
        ">
            {texto_resposta}
        </p>

    </div>

    """


    # RETORNO DOS RESULTADOS

    return (
        intencao_detectada,
        percentual_confianca,
        classificacao_status,
        card_resposta
    )


# BLOCO 5: INTERFACE GRÁFICA COM GRADIO


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SAC Imobiliário"
) as app:


    # CABEÇALHO

    gr.Markdown(
        """
        # SAC Imobiliário — ChatBot Inteligente

        *Será um prazer atendê-lo.
        Digite a seguir a sua necessidade.*
        """
    )


    # LINHA PRINCIPAL

    with gr.Row():



        # COLUNA DA ESQUERDA
        # ENTRADA DO USUÁRIO

        with gr.Column(scale=1):

            gr.Markdown(
                "### Mensagem do Cliente"
            )


            input_texto = gr.Textbox(
                lines=4,
                placeholder=(
                    "Ex: Preciso da segunda via "
                    "do boleto de aluguel..."
                ),
                label="Digite sua necessidade"
            )


            btn_processar = gr.Button(
                "Processar Mensagem",
                variant="primary",
                size="lg"
            )


            # EXEMPLOS

            gr.Examples(

                examples=[

                    [
                        "Preciso de suporte técnico "
                        "para consertar vazamento."
                    ],

                    [
                        "Quero ver apartamentos "
                        "à venda na zona sul."
                    ],

                    [
                        "Como faço para alugar "
                        "um galpão comercial?"
                    ],

                    [
                        "Gostaria de baixar o "
                        "boleto do condomínio."
                    ],

                    [
                        "Vocês vendem terreno "
                        "na Lua ou em Marte?"
                    ]

                ],

                inputs=input_texto
            )


        # COLUNA DA DIREITA
        # PAINEL DE DIAGNÓSTICO

        with gr.Column(scale=1):

            gr.Markdown(
                "### Painel de Diagnóstico"
            )


            # INDICADORES

            with gr.Row():

                out_intencao = gr.Textbox(
                    label="Intenção",
                    scale=2,
                    interactive=False
                )


                out_confianca = gr.Textbox(
                    label="Confiança",
                    scale=1,
                    interactive=False
                )


            # STATUS


            out_status = gr.Textbox(
                label="Status da Decisão",
                interactive=False
            )


            # RESPOSTA

            out_resposta = gr.HTML(

                value=(
                    "<div style='padding: 15px; "
                    "color: #888;'>"
                    "Aguardando envio de mensagem..."
                    "</div>"
                ),

                label="Resposta da Imobiliária"
            )


    # AÇÃO DO BOTÃO

    btn_processar.click(

        fn=processar_atendimento_sac,

        inputs=[
            input_texto
        ],

        outputs=[
            out_intencao,
            out_confianca,
            out_status,
            out_resposta
        ]
    )


# EXECUTAR A APLICAÇÃO


app.launch(
    debug=True,
    share=True
)


Carregando modelo morfológico SpaCy (pt_core_news_sm)
Carregando espaço vetorial denso de embeddings modelo GloVe
[==================================================] 100.0% 66.0/66.0MB downloaded
Dataset carregado com 20 mensagens divididas em 4 intenções.
Dimensão da matriz X: (20, 50)
Dimensão do vetor y: (20,)
Modelo supervisionado treinado com Árvore de Decisão!


/tmp/ipykernel_2257/949714679.py:541: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39c373f776d1df12b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
